# 2.2 三個位置怎樣一起送進猜字計算？


假設前文是「我想喝」，我們想讓三字共同影響下一字。先約定 ID 1=我、2=想、3=喝，手動給它們兩個特徵：`[1,10]`、`[2,20]`、`[3,30]`。這些數字只是為了追蹤，不是在說「我」有十倍某種意思。

最直接的方法是按位置把三排接成一條長卡：

`我｜想｜喝 → [1,10｜2,20｜3,30]`

這叫**拼接**。如果換成「喝想我」，得到 `[3,30｜2,20｜1,10]`。數字沒有相加，順序沒有丟掉。後面的配方可為長卡每格配置不同乘數，因此第一個位置和最後一個位置能有不同作用。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

embedding = nn.Embedding(4, 2)
with torch.no_grad():
    embedding.weight.copy_(torch.tensor([[0.0, 0.0], [1.0, 10.0], [2.0, 20.0], [3.0, 30.0]]))
ids = torch.tensor([[1, 2, 3], [3, 2, 1]])
x = embedding(ids)
context = x.reshape(2, -1)
print(x.shape)
print(context)
assert context.shape == (2, 6)

輸入有兩筆、每筆三位置，形狀 `[2,3]`；查表後每位置多兩個特徵，成 `[2,3,2]`。`reshape(2,-1)` 保留兩筆，把每筆六格依原順序排成一軸，輸出是：

```text
[1,10,2,20,3,30]
[3,30,2,20,1,10]
```

這裡 `-1` 要求按總格數推算長度：12 格分兩筆，就是每筆 6 格。reshape 只改排列的形狀，沒有產生新資訊。

如果筆數是 B、每筆看 C 個位置、每位置 D 格，便是 `[B,C,D]→[B,C×D]`。這次 B=2、C=3、D=2。C 越大，後面配方收到的格數越多，因此這種做法要先規定固定視窗長度；三字與五字長卡無法直接交給同一個固定輸入大小的配方。

練習只把第一筆改成 `[2,1,3]`，應得 `[2,20,1,10,3,30]`，第二筆不變。先逐段寫出再執行；光看總和無法分辨這次順序變化。

<details>
<summary>補充與重做</summary>

`no_grad()` 只包住手動填表，離開區塊後查表與拼接仍記錄求導關係，見 [1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)。顯示的 `grad_fn=<ViewBackward0>` 是 reshape 的求導記錄；本節沒有求導或更新。視窗不足時的補齊與過長時的擷取，後面再處理。

</details>
